# 📺 Abrir o app EPTV no Colab
1. Guarde a chave em 🔑 **Secrets**: `GROQ_API_KEY` (e, opcional, `ANTHROPIC_API_KEY`), com *Acesso ao notebook* ativado.
2. Rode as 3 células em ordem. Na célula 2, envie o arquivo **eptv_app.zip**.
3. Na célula 3 aparece um link `https://....trycloudflare.com`: é o app, pode abrir em outra aba e compartilhar enquanto o notebook estiver rodando.

In [ ]:
!pip -q install "streamlit>=1.50" groq pandas pymupdf pytesseract pillow anthropic
!apt-get -qq install -y tesseract-ocr tesseract-ocr-por > /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
from google.colab import files
import zipfile, os
for nome, conteudo in files.upload().items():
    open(nome, "wb").write(conteudo)
    zipfile.ZipFile(nome).extractall("/content")
os.listdir("/content/eptv_app")

In [ ]:
import os, subprocess, time, re
from google.colab import userdata

os.chdir("/content/eptv_app")
os.makedirs(".streamlit", exist_ok=True)
linhas = []
for nome in ("GROQ_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        linhas.append(f'{nome} = "{userdata.get(nome)}"')
    except Exception:
        pass
open(".streamlit/secrets.toml", "w").write("\n".join(linhas))

subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
subprocess.Popen("streamlit run app.py --server.port 8501 > streamlit.log 2>&1", shell=True)
time.sleep(6)
tun = subprocess.Popen("cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1", shell=True)
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        print("Abra o app aqui:", m.group(0)); break
else:
    print("Não consegui o link. Veja tunnel.log")